In [1]:
import pandas as pd
import numpy as np
import json
import os
import ast
from pathlib import Path
import torch
from typing import List, Optional
from dataclasses import dataclass

import teradatasql
from sqlalchemy import text, create_engine
from teradataml import create_context, get_context, get_connection, DataFrame, in_schema, copy_to_sql
from teradataml.dataframe.copy_to import copy_to_sql
from dotenv import load_dotenv

# import sys
# sys.path.append('..')
from constants import (
    CLEANED_TEST_DATA_PATH,
    ENCODED_TEST_DATA_PATH,
    CLEANED_TRAIN_DATA_PATH
)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\mk255155\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
# load_dotenv('../.env')

# TD_HOST = os.getenv('TD_HOST')
# TD_USER = os.getenv('TD_USER')
# TD_PASS = os.getenv('TD_PASS')
# TD_DB = os.getenv('TD_DB')

TD_HOST="iteration7-w9og53takluu3v27.env.clearscape.teradata.com"
TD_USER="demo_user"
TD_PASS="n8888888"
TD_DB="DEMO_USER"

In [ ]:
# print(TD_DB,TD_HOST,TD_PASS,TD_USER)
# conn = teradatasql.connect(
#     host=TD_HOST,
#     user=TD_USER,
#     password=TD_PASS,
#     database=TD_DB
# )
# cursor = conn.cursor()
# print("Successfully connected to Teradata!")

In [3]:
conn = teradatasql.connect(
    host=TD_HOST,
    user=TD_USER,
    password=TD_PASS,
    logdata={'CHARSET': 'UTF8'}
)

sqlalchemy_engine = create_engine("teradatasql://", creator=lambda: conn)
create_context(tdsqlengine=sqlalchemy_engine)
print("Connection successful with UTF-8 encoding!")

Connection successful with UTF-8 encoding!


In [4]:
tables_df = DataFrame.from_query(f"""
    SELECT DatabaseName, TableName
    FROM DBC.TablesV
    WHERE DatabaseName = '{TD_DB}'
""")

tables_df

DataBaseName,TableName
demo_user,ml___frmqry_v_1755601679762493
demo_user,original_labels_lookup
demo_user,remove_data
demo_user,ml___frmqry_v_1755608816397687
demo_user,cleaned_tdf_tmp
demo_user,ml__select__1755601586193745
demo_user,ml___frmqry_v_1755605247458733
demo_user,ml___frmqry_v_1755598922818827
demo_user,train_embeddings_fc
demo_user,original-dataset


In [5]:
tdf = DataFrame.from_table("original_dataset", schema_name=TD_DB, index_label="Item_Name")
print("Shape of the data:", tdf.shape)

Shape of the data: (4773, 10)


In [6]:
tdf.head(10)

Item_Name,class,Brand,Weight,Number of units,Size of units,Price,T.Price,Pack,Unit
أبو كاس أرز مزة بسمتي هندي 10 كجم,"Rice, Pasta & Pulses",أبو كاس,10كجم,1,None,None,None,كيس,كجم
أجنحة دجاج أطياب - 700جم,Poultry,أطياب,700جم,1,None,None,None,عبوة,جم
أجنحة دجاج حارة أطياب,Poultry,أطياب,None,1,None,None,None,عبوة,None
أحمد تي شاي أخضر نقي - 20 فتلة,"Tea, Coffee & Hot Drinks",أحمد تي,None,20,None,None,None,عبوة,None
أحمد تي شاي إيرل جراي - 25 فتلة,"Tea, Coffee & Hot Drinks",أحمد تي,None,25,None,None,None,عبوة,None
أحمد تي كولد برو شاي مثلج بالليمون والنعناع 20 كيس,"Tea, Coffee & Hot Drinks",احمد تي,None,20,None,None,None,None,كيس
أحمد تي شاي إيرل جراي - 100 فتلة,"Tea, Coffee & Hot Drinks",أحمد تي,100جم,1,None,None,None,علبة,جم
أجرومونتي صلصة الداترينو 330 جم,"Tins, Jars & Packets",أجرومونتي,330جم,1,None,None,None,عبوة,جم
أبو عوف قهوة تركي محوج وسط 250 جم,"Tea, Coffee & Hot Drinks",أبو عوف,250جم,1,None,None,None,عبوة,جم
أبو علي بابريكا - 85جم,Cooking Ingredients,أبو علي,85جم,1,None,None,None,عبوة,جم


In [ ]:
tdf.tdtypes

In [7]:
tdf = tdf.dropna(subset=["Item_Name", "class"])

In [ ]:
tdf.count()

In [9]:
tdf = tdf.assign(
    Item_Name = tdf.Item_Name.str.lower(),
    **{'class': tdf['class'].str.lower()}
)

In [10]:
tdf_stripped = tdf.assign(
    Item_Name = tdf.Item_Name.str.strip(),
    **{'class': tdf['class'].str.strip()}
)

In [11]:
tdf = tdf_stripped.assign(
    Item_Name = tdf_stripped.Item_Name.otranslate("!@#$%^&*()-_=+[]{};:',.<>?/\\|`~", ""),
    **{'class': tdf_stripped['class'].otranslate("!@#$%^&*()-_=+[]{};:',.<>?/\\|`~", "")}
)

In [13]:
cleaned_tdf = tdf[["Item_Name", "class"]]
cleaned_tdf

Item_Name,class
صلصه هاينز برطمان خصم عرض,tins jars packets
بودرة عصير أناناس من سورس، 900 جم,soft drinks juices
بسكو مصر لوكس 6 قطعه علبه 12,biscuits cakes
anise 100g,tea coffee hot drinks
مسحوق برسيل جيل باللافندر 39 كجم,cleaning supplies
شكولاته الشمعدان بيور بندق,biscuits cakes
لوبيا بلدى 500 جم,rice pasta pulses
dasani water 330ml,water
ليمون اداليا 500 جم,tins jars packets
americana okra zero 400 gm,vegetables fruits


In [ ]:
from teradataml.analytics.analytic_functions import RowNumber

# Assuming 'cleaned_tdf' is your existing teradataml DataFrame with 2 columns.

# Get the name of the first column to order by. This ensures the row numbers are generated consistently.
# You can replace this with any column name you prefer for ordering.
order_by_column = cleaned_tdf.columns[0]

# 1. Add the new 'row_id' column using assign() and RowNumber()
# The data is not moved; a new DataFrame object with the added step in its SQL query is created.
tdf_with_id = cleaned_tdf.assign(
    row_id=RowNumber(order_by=order_by_column)
)

# 2. Reorder the columns to put 'row_id' first
# Get the original column names
original_columns = cleaned_tdf.columns

# Create the new column order
new_column_order = ['row_id'] + original_columns

# Apply the new order
cleaned_tdf_with_row_id = tdf_with_id[new_column_order]

# Display the first 10 rows of the new DataFrame to verify
print(cleaned_tdf_with_row_id.head(10))

In [15]:
from teradataml import execute_sql
from teradatasqlalchemy.types import VARCHAR

target_table_name = 'DEMO_USER.cleaned_data'

unicode_type = VARCHAR(length=500, charset='UNICODE')

unicode_tdf = cleaned_tdf.assign(
    Item_Name = cleaned_tdf.Item_Name.cast(unicode_type)
)

source_query = unicode_tdf.show_query()

create_sql = f"""
CREATE TABLE {target_table_name} AS (
    {source_query}
) WITH DATA;
"""

print("--- Generated SQL ---")
print(create_sql)

try:
    print(f"\nAttempting to drop existing table '{target_table_name}'...")
    execute_sql(f"DROP TABLE {target_table_name};")
    print("✔ Previous table dropped.")
except Exception:
    print("Table did not exist, proceeding to create.")

print("\nExecuting CREATE TABLE statement... 🚀")
execute_sql(create_sql)
print(f"✔ Success! Table '{target_table_name}' has been created.")

--- Generated SQL ---

CREATE TABLE DEMO_USER.cleaned_data AS (
    select CAST("Item_Name" AS VARCHAR(500) CHAR SET UNICODE) AS "Item_Name", "class" AS "class" from (select "Item_Name","class" from (select OTRANSLATE("Item_Name", '!@#$%^&*()-_=+[]{};:'',.<>?/\|`~', '') AS "Item_Name", OTRANSLATE("class", '!@#$%^&*()-_=+[]{};:'',.<>?/\|`~', '') AS "class", "Brand" AS "Brand", "Weight" AS "Weight", "Number of units" AS "Number of units", "Size of units" AS "Size of units", "Price" AS "Price", "T.Price" AS "T.Price", "Pack" AS "Pack", "Unit" AS "Unit" from (select rtrim(ltrim("Item_Name", '
'), '
') AS "Item_Name", rtrim(ltrim("class", '
'), '
') AS "class", "Brand" AS "Brand", "Weight" AS "Weight", "Number of units" AS "Number of units", "Size of units" AS "Size of units", "Price" AS "Price", "T.Price" AS "T.Price", "Pack" AS "Pack", "Unit" AS "Unit" from (select lower("Item_Name") AS "Item_Name", lower("class") AS "class", "Brand" AS "Brand", "Weight" AS "Weight", "Number of un

In [17]:
tdf_2 = DataFrame.from_table("cleaned_data", schema_name=TD_DB)
print("Shape of the data:", tdf_2.shape)
tdf_2.head(5)

Shape of the data: (4772, 2)


Item_Name,class
أبو كاس أرز مزة بسمتي هندي 10 كجم,rice pasta pulses
أجنحة دجاج أطياب 700جم,poultry
أجرومونتي صلصة الداترينو 330 جم,tins jars packets
أبو عوف قهوة تركي محوج وسط 250 جم,tea coffee hot drinks
أبو علي بابريكا 85جم,cooking ingredients


In [ ]:
tables_df = DataFrame.from_query(f"""
    SELECT DatabaseName, TableName
    FROM DBC.TablesV
    WHERE DatabaseName = '{TD_DB}'
""")

tables_df

In [ ]:
## Verify unicode on teradata session

# td_context = get_context()
# raw_connection = td_context.engine.raw_connection()
# print("Verifying the character set of the established session...")
# with raw_connection.cursor() as cur:
#     cur.execute("HELP SESSION;")
    
#     session_info = cur.fetchone()
    
#     if session_info:
#         character_set = session_info[5]
#         print(f"✅ Session Character Set is: {character_set}")
#     else:
#         print("❌ Could not retrieve session information.")

# raw_connection.close()

In [ ]:
# copy_to_sql(
#     df=cleaned_tdf,              
#     table_name="cleaned_data",
#     if_exists="replace"
# )

In [ ]:
# tdf_2 =  tdf[['Item_Name', 'class']]
# tdf_2
######
# cat=tdf.groupby(['class']).count()
# cat
#######

In [ ]:
## cleaning
# tdf = tdf.assign(Item_Name = tdf.Item_Name.str.lower())
# tdf_stripped = tdf.assign(Item_Name = tdf.Item_Name.str.strip())

# tdf = tdf_stripped.assign(
#     Item_Name = tdf.Item_Name.otranslate("!@#$%^&*()-_=+[]{};:',.<>?/\\|`~", "")
# )

In [ ]:
# remove_context()